In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
from models.cnn_model import CNN_Model
from utils import (
    EarlyStopping,
    Face_Detector,
    ResizeKeepRatioPad,
    get_device,
    get_mean_std,
    create_preprocess_config,
    build_cnn_transform,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
import torchvision.transforms as transforms

# Parameters

In [ ]:
batch_size   = 64
epochs       = 20
hidden_size  = 16
kernel_size  = 3
stride       = 1
padding      = 1
pool_size    = 2
input_size   = (80, 80)
patience     = 5
test_size    = 0.2
lr = 0.001

is_face = True
train_data_path = '../data/face/train/'

In [4]:
model_output_filename = 'cnn_checkpoint'

# Sample data

In [ ]:
stats_steps = []
if is_face:
    stats_steps.append(Face_Detector(input_size=input_size))

stats_steps.extend([
    ResizeKeepRatioPad(input_size, fill=0),
    transforms.ToTensor(),
])

stats_transform = transforms.Compose(stats_steps)

## Option 1 : MNIST dataset

In [6]:
# train_dataset = torchvision.datasets.MNIST(
#     root="../data",
#     train=True,
#     download=True,
#     transform=base_transform,
# )

In [7]:
# test_dataset = torchvision.datasets.MNIST(
#     root="../data",
#     train=False,
#     download=True,
#     transform=base_transform,
# )

## Option 2 : Local dataset

In [ ]:
full_dataset = datasets.ImageFolder(
    
    root = train_data_path,
    transform=stats_transform,    
)

In [9]:
class_names = full_dataset.classes
num_output = len(class_names)


test_records = int(test_size * len(full_dataset) )
train_records = len(full_dataset) - test_records


train_dataset, test_dataset  = random_split(    
    full_dataset, 
    [train_records,test_records],
    generator=torch.Generator().manual_seed(42)
)


# Data Processing

## Get number of channels (RGB / Gray) and statistic of dataset

In [ ]:
sample_x, _ = train_dataset[0]
channel_size = sample_x.shape[0]


mean, std = get_mean_std(train_dataset)

print(f"Detected channel_size: {channel_size}")
print(f"mean: {mean}, std: {std}")
print(f"class_names: {class_names}")

o test 3
o test 3
o test 3
o test 3
o test 3
o test 3
o test 3
o test 3
o test 3
Face detection failed: list index out of range, returning original image
o test 3
Face detection failed: list index out of range, returning original image
Face detection failed: list index out of range, returning original image
o test 3
o test 3
o test 3
Face detection failed: list index out of range, returning original image
Face detection failed: list index out of range, returning original image
o test 3
o test 3
o test 3
o test 3
Face detection failed: list index out of range, returning original image
o test 3
o test 3
o test 3
o test 3
Face detection failed: list index out of range, returning original image
Face detection failed: list index out of range, returning original image
o test 3
o test 3
o test 3
o test 3
o test 3
o test 3
o test 3
Face detection failed: list index out of range, returning original image
o test 3
Face detection failed: list index out of range, returning original image
Face dete

## Prepare config of the model

In [ ]:
preprocess_config = create_preprocess_config(
    input_size=input_size,
    mean=mean,
    std=std,
    channel_size=channel_size,
    keep_ratio_pad=True,
    pad_fill=0,
)

In [ ]:
model_config = {
    "channel_size": channel_size,
    "input_size": tuple(input_size),
    "hidden_size": hidden_size,
    "num_output": num_output,
    "kernel_size": kernel_size,
    "stride": stride,
    "padding": padding,
    "pool_size": pool_size,
}

## Normalize data

In [ ]:
norm_transform = build_cnn_transform(preprocess_config)

final_steps = []
if is_face:
    final_steps.append(Face_Detector(input_size=input_size))

final_steps.extend(norm_transform.transforms)
final_transform = transforms.Compose(final_steps)

In [ ]:
train_dataset.dataset.transform = final_transform
test_dataset.dataset.transform = final_transform

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Prepare model 

In [ ]:
device = get_device()

In [ ]:
model = CNN_Model(
    channel_size=channel_size,
    input_size=input_size,
    hidden_size=hidden_size,
    num_output=num_output,
    kernel_size=kernel_size,
    stride=stride,
    padding=padding,
    pool_size=pool_size,
)

In [ ]:
_ = model.to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()



early_stopping = EarlyStopping(
    patience=patience,
    path=f"../checkpoints/{model_output_filename}.pt",
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "class_names": class_names,
    },
)

# Loop epochs and train model

In [ ]:
for epoch in range(epochs):

    # ----------------------------- Train -----------------------------
    model.train()
    train_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ------------------- Validation -------------------

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            output = model(batch_x)
            loss = criterion(output, batch_y)
            val_loss += loss.item()

            preds = output.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)

    avg_val_loss = val_loss / len(test_loader)
    accuracy = correct / total

    print(
        f"Epoch {epoch + 1:3d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Accuracy: {accuracy:.4f}"
    )

    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        print("Early stopping triggered! Training stopped.")
        break

print("Training finished!")